# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I used a line chart because it shows change over time. I set every continent to 100 in 1970 so growth is easy to compare, and only Africa is coloured so it stands out.

In [2]:
# Q1
# YOUR CODE HERE
ACCENT, GREY = '#E07B39', '#BBBBBB'   # orange accent + grey context, reused in all questions

# Step 1: Total population per continent and year
cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum().sort_values(['Continent', 'Year'])

# Step 2: Index each continent to its own 1970 value (1970 = 100)
base = cont[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Index'] = cont['Population'] / cont['Continent'].map(base) * 100
end = cont[cont['Year'] == 2022].sort_values('Index', ascending=False)
fastest = end.iloc[0]

# Step 3: Grey context lines, one accent line for the fastest grower
fig = px.line(cont, x='Year', y='Index', color='Continent',
              color_discrete_map={c: (ACCENT if c == fastest['Continent'] else GREY) for c in cont['Continent'].unique()},
              category_orders={'Continent': [c for c in end['Continent'] if c != fastest['Continent']] + [fastest['Continent']]},
              template='plotly_white', height=550)
fig.update_traces(line_width=2)
fig.update_traces(line_width=4, selector=dict(name=fastest['Continent']))

# Step 4: Direct labels — the accent line, Europe (the slowest) and one label for the middle group
labels = {fastest['Continent']: ACCENT, 'Europe': '#888888'}
for name, col in labels.items():
    r = end[end['Continent'] == name].iloc[0]
    fig.add_annotation(x=2022, y=r['Index'], text=f"{name} {r['Index']/100:.1f}×", showarrow=False,
                       xanchor='left', xshift=6, font=dict(color=col, size=13))
mid = end[~end['Continent'].isin(labels)]
fig.add_annotation(x=2022, y=mid['Index'].mean(), showarrow=False, xanchor='left', xshift=6,
                   text=f"Other continents {mid['Index'].min()/100:.1f}–{mid['Index'].max()/100:.1f}×",
                   font=dict(color='#888888', size=12))

# Step 5: Layout
fig.update_layout(
    title=f"{fastest['Continent']}'s population is {fastest['Index']/100:.1f}× its 1970 size — no other continent comes close",
    showlegend=False, font=dict(family='Arial', size=13), margin=dict(r=170))
fig.update_xaxes(title='', showgrid=False, range=[1969, 2023])
fig.update_yaxes(title='Population (1970 = 100)', showgrid=False, rangemode='tozero')
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I used a sorted horizontal bar chart because it is best for ranking. African countries are orange, so you can count them quickly.

In [3]:
# Q2
# YOUR CODE HERE

# Step 1: People added 1970 -> 2022, top 15
df['Added'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Added').copy()
n_africa = (top15['Continent'] == 'Africa').sum()

# Step 2: Colour Africa, grey the rest; label in millions
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')
top15['Label'] = (top15['Added'] / 1e6).map('{:,.0f}M'.format)

fig = px.bar(top15.sort_values('Added'), x='Added', y='Country', orientation='h',
             color='Group', text='Label',
             color_discrete_map={'Africa': ACCENT, 'Other continents': GREY},
             category_orders={'Country': top15.sort_values('Added')['Country'].tolist()},
             template='plotly_white', height=600)
fig.update_traces(textposition='outside', cliponaxis=False)

# Step 3: Layout — axis hidden because every bar has its own label
fig.update_layout(
    title=f"India and China dominate, but {n_africa} of the 15 biggest population gainers since 1970 are in Africa",
    font=dict(family='Arial', size=13), bargap=0.25,
    legend=dict(title='', orientation='h', x=0, y=1.05))
fig.update_xaxes(visible=False)
fig.update_yaxes(title='')
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*


I used a scatter plot because it shows if area and population are related. I used log scales because the numbers are very different in size, and I labelled the countries that break the pattern.

In [4]:
# Q3
# YOUR CODE HERE

# Step 1: Scatter on log-log axes (both variables span many orders of magnitude)
fig = px.scatter(df, x='Area', y='2022 Population', color='Continent', hover_name='Country',
                 log_x=True, log_y=True,
                 color_discrete_sequence=px.colors.qualitative.Safe,   # CVD-safe palette
                 labels={'Area': 'Land area (km², log)', '2022 Population': 'Population 2022 (log)'},
                 template='plotly_white', height=600)
fig.update_traces(marker=dict(size=8, opacity=0.75))

# Step 2: Label the countries that break the pattern (log axes -> log10 positions)
outliers = {'Bangladesh': (-60, -30), 'Singapore': (-50, -25), 'Greenland': (0, 40),
            'Mongolia': (40, 35), 'Australia': (45, 30), 'Monaco': (40, -25)}
for name, (ax, ay) in outliers.items():
    r = df[df['Country'] == name].iloc[0]
    fig.add_annotation(x=np.log10(r['Area']), y=np.log10(r['2022 Population']), text=name,
                       showarrow=True, arrowhead=0, arrowcolor='#888888', ax=ax, ay=ay, font=dict(size=11))

# Step 3: Layout
fig.update_layout(
    title='Larger countries usually have more people — Bangladesh and Singapore are far denser, Greenland and Mongolia far emptier',
    font=dict(family='Arial', size=13),
    legend=dict(title='', orientation='h', x=0, y=1.06), margin=dict(t=100))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
I used a box plot because it shows the spread, not just the average. I used a log scale because density varies a lot between countries.

In [5]:
# Q4
# YOUR CODE HERE
# Step 1: Order continents by median density
med = df.groupby('Continent')['Density'].median().sort_values()
order = med.index.tolist()

# Step 2: Horizontal box plot, log axis (density is extremely skewed)
fig = px.box(df, x='Density', y='Continent', orientation='h', points='outliers',
             hover_name='Country', log_x=True,
             category_orders={'Continent': order[::-1]},   # densest median on top
             color_discrete_sequence=['#2E75B6'],
             labels={'Density': 'People per km² (log scale)', 'Continent': ''},
             template='plotly_white', height=500)

# Step 3: Layout
fig.update_layout(
    title=(f"North America is the most crowded continent (median {med.iloc[-1]:.0f}/km²), "
           f"South America the emptiest ({med.iloc[0]:.0f}/km²)"),
    font=dict(family='Arial', size=13))
fig.update_xaxes(showgrid=False)
fig.show()



---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
I used a map because the question asks where in the world. Blue means the population grew, red means it shrank, and white means no change.

In [6]:
# Q5
# YOUR CODE HERE
# Step 1: % change 2000 -> 2022
df['Change_pct'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100
shrank = (df['Change_pct'] < 0).sum()

# Step 2: Choropleth, diverging scale centred on 0 (shrink vs grow)
fig = px.choropleth(df, locations='CCA3', locationmode='ISO-3', color='Change_pct',
                    hover_name='Country', hover_data={'CCA3': False, 'Change_pct': ':+.0f'},
                    color_continuous_scale='RdBu', color_continuous_midpoint=0,
                    range_color=[-50, 50],                      # symmetric; extremes like Qatar capped
                    labels={'Change_pct': 'Change<br>2000–22 (%)'},
                    template='plotly_white', height=550)

# Step 3: Layout
fig.update_geos(showframe=False, showcoastlines=False, projection_type='natural earth')
fig.update_layout(
    title=f"Africa and the Middle East grew fastest since 2000 — {shrank} countries shrank, about half of them in Europe",
    font=dict(family='Arial', size=13), margin=dict(l=0, r=0, t=60, b=0))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
I used a treemap because it shows continents and their countries in one chart. A bigger box means more people.

In [7]:
# Q6
# YOUR CODE HERE
# Step 1: Numbers for the title
total = df['2022 Population'].sum()
asia_share = df.loc[df['Continent'] == 'Asia', '2022 Population'].sum() / total
ci_share = df.loc[df['Country'].isin(['China', 'India']), '2022 Population'].sum() / total

# Step 2: Treemap World -> Continent -> Country
fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_sequence=px.colors.qualitative.Safe,
                 color_discrete_map={'(?)': '#DDDDDD'},      # grey root node
                 template='plotly_white', height=650)
fig.update_traces(texttemplate='%{label}<br>%{percentRoot:.1%}', root_color='white',
                  hovertemplate='<b>%{label}</b><br>%{value:,.0f} people<br>%{percentRoot:.1%} of world<extra></extra>')

# Step 3: Layout
fig.update_layout(title=f"Asia is home to {asia_share:.0%} of the world — China and India together hold {ci_share:.0%}",
                  font=dict(family='Arial', size=13), margin=dict(t=60, l=10, r=10, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I used a 100% stacked bar chart because it shows how the world population is split and how that changed over time. Africa is orange at the bottom, so its rise is easy to see.

In [8]:
# Q7
# YOUR CODE HERE

# Step 1: Group into 4 regions and compute each year's shares
df_long['Region'] = np.where(df_long['Continent'].isin(['Africa', 'Asia', 'Europe']),
                             df_long['Continent'], 'Rest of the world')
sh = df_long.groupby(['Year', 'Region'], as_index=False)['Population'].sum()
sh['Share'] = sh['Population'] / sh.groupby('Year')['Population'].transform('sum')

# Step 2: Numbers for the title
a70 = sh.query("Year == 1970 and Region == 'Africa'")['Share'].iloc[0]
a22 = sh.query("Year == 2022 and Region == 'Africa'")['Share'].iloc[0]
e70 = sh.query("Year == 1970 and Region == 'Europe'")['Share'].iloc[0]
e22 = sh.query("Year == 2022 and Region == 'Europe'")['Share'].iloc[0]

# Step 3: Labels only on Africa and Europe
sh['Year'] = sh['Year'].astype(str)
sh['Label'] = np.where(sh['Region'].isin(['Africa', 'Europe']), sh['Share'].map('{:.0%}'.format), '')

# Step 4: 100% stacked bar — Africa on the baseline
fig = px.bar(sh, x='Year', y='Share', color='Region', text='Label',
             category_orders={'Region': ['Africa', 'Europe', 'Asia', 'Rest of the world']},
             color_discrete_map={'Africa': ACCENT, 'Europe': '#2E75B6',
                                 'Asia': '#BBBBBB', 'Rest of the world': '#E3E3E3'},
             template='plotly_white', height=550)
fig.update_traces(textposition='inside', textfont=dict(color='white', size=12))

# Step 5: Layout
fig.update_layout(
    title=f"Africa's share of humanity rose from {a70:.0%} to {a22:.0%} since 1970 — Europe's fell from {e70:.0%} to {e22:.0%}",
    font=dict(family='Arial', size=13), bargap=0.25,
    legend=dict(title='', orientation='h', x=0, y=1.06, traceorder='normal'))
fig.update_yaxes(visible=False)
fig.update_xaxes(title='')
fig.show()